## 1. Environment Setup

Confirm CUDA / GPU availability before doing anything expensive.

In [1]:
# GPU setup
import torch

# Check CUDA availability and GPU properties
print(f"torch version: {torch.__version__}")
print(f"CUDA version: {torch.version.cuda}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU count: {torch.cuda.device_count()}")

if torch.cuda.is_available():
    print(f"GPU name: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

torch version: 2.11.0+cu128
CUDA version: 12.8
CUDA available: True
GPU count: 1
GPU name: NVIDIA GeForce RTX 3060
VRAM: 12.9 GB


## 2. Base Model & Tokenizer Loading

In [1]:
# Load the model and tokenizer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, BitsAndBytesConfig
import torch
save_directory = "./models/flan-t5-base-local"

# Load from local directory
tokenizer = AutoTokenizer.from_pretrained(save_directory, local_files_only=True)
model = AutoModelForSeq2SeqLM.from_pretrained(
    save_directory,
    device_map="auto",           # Automatically place model layers
    trust_remote_code=True,      # Required for some models
    dtype=torch.bfloat16,        # Use bfloat16 for non-quantized parts
)


c:\Users\Youssef\Desktop\slm_fine_tune\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 282/282 [00:00<00:00, 784.80it/s] 
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


## 3. Dataset Loading & Tokenization


In [2]:
# Load the datasets
from datasets import load_dataset

dataset = load_dataset(
    "json",
    data_files={
        "train": "totto_data/train.jsonl",
        "validation": "totto_data/validation.jsonl"
    }
)

print(dataset["train"][0])
print(dataset["validation"][0])

Generating train split: 92252 examples [00:00, 347799.88 examples/s]
Generating validation split: 5854 examples [00:00, 307822.42 examples/s]


{'id': 1762238357686640028, 'prompt': 'Task:\nGenerate a single factual sentence describing the information contained in the highlighted cells.\nUse only information provided below.\nDo not invent facts.\nPage Title:\nList of 8/9 PM telenovelas of Rede Globo\nSection Title:\n2000s\nHighlighted Cells:\n(13,2) (Title)\nTable:\n[0] # | Run | Title | Chapters | Author | Director | Ibope Rating\n...\n[10] 68 | July 10, 2006— March 2, 2007 | Páginas da Vida | 203 | Manoel Carlos | Jayme Monjardim | 46.8\n[11] 69 | March 5, 2007— September 28, 2007 | Paraíso Tropical | 179 | Gilberto Braga Ricardo Linhares | Dennis Carvalho | 42.8\n[12] 70 | October 1, 2007— May 31, 2008 | Duas Caras | 210 | Aguinaldo Silva | Wolf Maya | 41.1\n[13] 71 | June 2, 2008— January 16, 2009 | (13,2)A Favorita | 197 | João Emanuel Carneiro | Ricardo Waddington | 39.5\n[14] 72 | January 19, 2009— September 11, 2009 | Caminho das Índias | 203 | Glória Perez | Marcos Schechtman | 38.8\n[15] 73 | September 14, 2009— May 

In [3]:
# NOTE: no explicit -100 masking of pad tokens is needed here - DataCollatorForSeq2Seq
# (instantiated in Section 6) pads labels with -100 automatically at batch time.
# Tokenize the dataset
def preprocess(examples):
    model_inputs = tokenizer(
        examples["prompt"],
        max_length=384,
        truncation=True,
    )

    labels = tokenizer(
        text_target=examples["target"],
        max_length=64,
        truncation=True,
    )

    model_inputs["labels"] = labels["input_ids"]

    return model_inputs

In [4]:
tokenized_dataset = dataset.map(
    preprocess,
    batched=True,
    remove_columns=dataset["train"].column_names
)

Map: 100%|██████████| 5854/5854 [00:00<00:00, 8550.70 examples/s]


In [5]:
# Stats
print(f"Number of training samples: {len(tokenized_dataset['train'])}")
print(f"Number of validation samples: {len(tokenized_dataset['validation'])}")

Number of training samples: 92252
Number of validation samples: 5854


## 4. Sequence Length Analysis

Before committing to `max_length=384` above, check whether that's actually a reasonable cutoff for this data.

In [6]:
import numpy as np

train_lengths = [len(x["input_ids"]) for x in tokenized_dataset["train"]]
print(f"Training lengths stats:")
print(f"Average length: {np.mean(train_lengths):.1f}")
print(f"Median length : {np.median(train_lengths):.1f}")
print(f"95th percentile: {np.percentile(train_lengths,95):.1f}")
print(f"Maximum length: {np.max(train_lengths)}")

print("*"*50)

validation_lengths = [len(x["input_ids"]) for x in tokenized_dataset["validation"]]

print(f"Validation lengths stats:")
print(f"Average length: {np.mean(validation_lengths):.1f}")
print(f"Median length : {np.median(validation_lengths):.1f}")
print(f"95th percentile: {np.percentile(validation_lengths,95):.1f}")
print(f"Maximum length: {np.max(validation_lengths)}")


Training lengths stats:
Average length: 245.6
Median length : 241.0
95th percentile: 363.0
Maximum length: 384
**************************************************
Validation lengths stats:
Average length: 245.5
Median length : 240.0
95th percentile: 363.0
Maximum length: 384


In [7]:
lengths = [
    len(tokenizer(x["prompt"]).input_ids)
    for x in dataset["train"]
]

import numpy as np

print("<=256 :", np.mean(np.array(lengths) <= 256))
print("<=384 :", np.mean(np.array(lengths) <= 384))
print("<=512 :", np.mean(np.array(lengths) <= 512))

<=256 : 0.5771473789186142
<=384 : 1.0
<=512 : 1.0


## 5. LoRA Configuration


In [8]:
# LoRA configuration
from peft import LoraConfig, TaskType, get_peft_model
peft_config = LoraConfig(
    r=8,                                # Rank of the low-rank matrices
    lora_alpha=16,                      # Scaling factor for the low-rank matrices
    target_modules=["q", "v"],          # Target modules for LoRA
    lora_dropout=0.1,                   # Dropout rate for LoRA layers
    bias="none",                        # No bias in LoRA layers
    task_type=TaskType.SEQ_2_SEQ_LM,    # Task type for encoder-decoder models
    )

model = get_peft_model(model, peft_config)

W0724 21:30:54.546000 18680 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


In [9]:
model.print_trainable_parameters()

trainable params: 884,736 || all params: 248,462,592 || trainable%: 0.3561


## 6. Training Configuration & Sanity Checks
### 6.1 Trainer Setup

In [10]:
# Trainer configuration
import os
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments, EarlyStoppingCallback, DataCollatorForSeq2Seq
os.environ["TENSORBOARD_LOGGING_DIR"] = "./logs"

training_args = Seq2SeqTrainingArguments(
    output_dir="./results",

    # Training
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,
    gradient_checkpointing=False,
    learning_rate=1e-4,
    weight_decay=0.01,
    warmup_steps=500,
    lr_scheduler_type="cosine",

    # Logging
    logging_steps=1,
    save_steps=1000,
    eval_steps=500,
    eval_strategy="epoch",
    save_strategy="epoch",
    train_sampling_strategy="group_by_length",
    logging_first_step=True,   
    logging_nan_inf_filter=False,

    # Generation
    predict_with_generate=False,
    generation_max_length=64,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    # Mixed precision
    fp16=False,
    bf16=True, 

    # Optimizer
    optim="adamw_torch", # for qlora paged_adamw_32bit

    seed=67,        # Set a random seed for reproducibility

    report_to="tensorboard",
)


data_collator = DataCollatorForSeq2Seq(
    tokenizer,
    model=model,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)
    

### 6.2 Pre-Training Sanity Check

Before committing to a full training run, confirm the pipeline produces a finite, reasonable loss on an untouched batch.

In [16]:
model.eval()
batch = next(iter(trainer.get_train_dataloader()))
batch = {k: v.to(model.device) for k, v in batch.items()}
with torch.no_grad():
    out = model(**batch)
print("loss on a fresh, untrained batch:", out.loss)

loss on a fresh, untrained batch: tensor(1.9922, device='cuda:0', dtype=torch.bfloat16)


## 7. Fine-Tuning


In [11]:
# Fine-tuning the model
trainer.train()

Epoch,Training Loss,Validation Loss
1,2.233918,1.086051
2,2.335529,1.057030
3,2.003266,1.053308


TrainOutput(global_step=8649, training_loss=2.5789714921539346, metrics={'train_runtime': 6368.7992, 'train_samples_per_second': 43.455, 'train_steps_per_second': 1.358, 'total_flos': 9.178293696878592e+16, 'train_loss': 2.5789714921539346, 'epoch': 3.0})

## 8. Save Fine-Tuned Adapter

In [12]:
# Save the fine-tuned model
model.save_pretrained("./models/flan-t5-base-totto-lora-finetuned-optimized")